# Cafe stores: average ticket and rating by region

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Labelled / Stratified Scatter Plot** · a worked example from the StatNex teaching guide

**The story.** A cafe chain has 47 stores in six regions. For each store it knows the average amount a customer spends (the average ticket) and the store's online customer rating.

**The question.** Do stores that charge more get better or worse ratings, and does the answer depend on the region?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/labelled_scatter_plot.csv` (47 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Store,Format,Region,Customer rating,Average ticket ($)
Northfield,Mall,Coast,4.61,10.64
Ashford,Drive-thru,City,4.04,8.62
Brookvale,High street,North,3.73,6.86
Cedarpoint,High street,South,4.01,8.41
Dunmore,High street,East,4.07,8.36
Elmwood,Drive-thru,West,4.22,7.38
Fairhaven,High street,Coast,4.31,9.64
Glenrock,Drive-thru,City,3.67,8.2
Harborcity,Mall,North,3.81,7.44
Ironbridge,High street,South,4.22,8.48
Juniper,Drive-thru,East,4.23,8.41
Kingsport,High street,West,3.89,7.66
Lakeside,Mall,Coast,4.48,10.4
Maplewood,Drive-thru,City,3.93,8.68
Newbury,Drive-thru,North,4.04,7.16
Oakridge,Mall,South,4.0,8.06
Pinecrest,Mall,East,4.19,8.67
Queensbay,High street,West,4.03,6.53
Riverton,Drive-thru,Coast,4.58,11.33
Silverton,Drive-thru,City,4.3,9.3
Thornbury,Drive-thru,North,4.25,7.16
Upton,Drive-thru,South,4.22,9.18
Valemont,Drive-thru,East,4.22,8.05
Westbrook,Drive-thru,West,4.0,7.07
Yarrow,High street,Coast,4.32,10.44
Zephyrbay,High street,City,3.6,7.38
Amberly,Drive-thru,North,3.69,6.8
Bellmont,High street,South,4.48,8.9
Clearwater,High street,East,3.76,6.93
Dovecliff,Mall,West,4.17,7.34
Eastwick,Mall,Coast,4.32,10.45
Foxhill,Mall,City,4.04,8.62
Greenhaven,Mall,North,4.01,7.69
Hollowell,Drive-thru,South,3.89,7.63
Ivydale,Mall,East,3.52,7.8
Jasperfalls,High street,West,4.14,7.16
Kestrelbay,Drive-thru,Coast,4.48,10.34
Larkspur,Mall,City,4.22,9.56
Millbrook,Drive-thru,North,3.82,6.9
Northgate,Drive-thru,South,4.38,8.68
Orchardhill,Mall,East,4.17,7.91
Primrose,High street,West,4.15,7.64
Quarryglen,Drive-thru,Coast,3.97,8.88
Rosewood,Drive-thru,City,4.13,6.59
Stonegate,Drive-thru,North,4.16,7.36
Tidewater,Mall,South,3.74,7.75
Willowby,Drive-thru,East,4.27,9.33
""")

## Libraries

In [ ]:
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per case. Text in X or Y counts as missing, and a case needs both. A case
# whose stratum is blank belongs to no group, so it is left out everywhere.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
x_name, y_name = "Average ticket ($)", "Customer rating"
label_name = "Store"
stratum_name = "Region"


def as_label(value):
    # A category as the sheet prints it: 3.0 read from the file is the label "3".
    if pd.isna(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def cut(text, limit=40):
    # Labels are cut at 40 half-width characters; a full-width character counts as two.
    width, kept = 0, ""
    for character in text:
        width += 2 if unicodedata.east_asian_width(character) in ("F", "W", "A") else 1
        if width > limit:
            break
        kept += character
    return kept


labels = data[label_name].map(as_label).map(cut)
strata = data[stratum_name].map(as_label)
x = pd.to_numeric(data[x_name], errors="coerce")
y = pd.to_numeric(data[y_name], errors="coerce")
usable = x.notna() & y.notna() & data[stratum_name].notna()


def cp932_key(text):
    # Groups in the order a Japanese Excel sorts them: by their bytes in Shift_JIS (CP932).
    try:
        return text.encode("cp932")
    except UnicodeEncodeError:
        return text.encode("utf-8")


groups = sorted({s for s in strata if s.strip()}, key=cp932_key)

## Data for the chart

In [ ]:
# Cases grouped by stratum (in the order above), keeping their order inside a group,
# with Y written in its own group's column -- one column per series.
cases = pd.DataFrame({"label": labels, "stratum": strata, "x": x, "y": y})[usable]
cases = cases.sort_values("stratum", key=lambda s: s.map(groups.index), kind="stable")
table = pd.DataFrame({"No.": np.arange(1, len(cases) + 1), label_name: cases["label"].to_numpy(),
                      stratum_name: cases["stratum"].to_numpy(), x_name: cases["x"].to_numpy()})
for name in groups:
    table[name] = np.where(cases["stratum"] == name, cases["y"], np.nan)
print(table.to_string(index=False))

## Correlation by stratum

In [ ]:
# Pearson's r inside each stratum and over all cases, with the P value of the t test of
# "no straight-line link" (it assumes independent cases and a roughly straight, evenly
# spread cloud). A stratum with fewer than 3 cases, or with a constant X or Y, has no
# coefficient: two points always give r = ±1. With only a handful of cases per
# stratum, r swings widely by chance, so read each stratum's P value before calling two
# strata different. If the strata share one slope but sit at different levels, the
# overall r mixes the link inside the strata with the gaps between them; when the two
# point in opposite directions, that is Simpson's paradox.
rows = []
for name, part in [*[(g, cases[cases["stratum"] == g]) for g in groups], ("Total", cases)]:
    if len(part) < 3 or part["x"].std() == 0 or part["y"].std() == 0:
        rows.append([name, len(part), np.nan, np.nan])
    else:
        r, p = stats.pearsonr(part["x"], part["y"])
        rows.append([name, len(part), r, p])
correlation = pd.DataFrame(rows, columns=["stratum", "n", "r", "P"])
print()
print(correlation.to_string(index=False))

## Scatter plot

In [ ]:
Path("figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 6))
for name in groups:
    part = cases[cases["stratum"] == name]
    ax.scatter(part["x"], part["y"], label=name, s=18)
    for _, case in part.iterrows():
        ax.annotate(case["label"], (case["x"], case["y"]), fontsize=7)
ax.set_xlabel(x_name)
ax.set_ylabel(y_name)
ax.legend(fontsize=8)
fig.savefig("figures/scatter.png", dpi=120, bbox_inches="tight")

## What the result says

The correlation coefficient (r) measures how closely two variables move together: 0 means no link and 1 means a perfect rising line. Across all 47 stores, r = 0.68 with a P value below 0.001. A P value is the chance of a link at least this strong if there were really none. So stores with a higher average ticket do get better ratings. The Coast stores lead on both.

Inside the regions, r runs from 0.08 in West (P = 0.87) to 0.88 in Coast (P = 0.004). But each region has only 7 or 8 stores.

So the link is clear for the chain as a whole. With so few stores per region, the chart cannot show whether the regions really differ.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.